# LLM Output Consistency Testing

**Same prompt, N runs — how consistent is your model?**

This notebook builds a reusable **consistency scorer** for LLM outputs. It runs one prompt
`N` times against a model (Groq or Gemini), then measures how stable the outputs are along
three axes:

1. **Semantic consistency** — embedding-based pairwise similarity between responses
2. **Structural consistency** — length variance, and agreement on the *final answer* (if the
   prompt has one, e.g. a math problem)
3. **Lexical diversity** — how much wording varies even when meaning stays similar

Built to slot straight into a Groq (`llama-3.1-8b-instant`) / Gemini (`gemini-2.5-flash-lite`)
workflow — same models used in CoT step-order perturbation-style experiments — so you can reuse
this as a sanity-check layer before running larger perturbation trials.

**Sections:** Install → Imports → API Config → Prompt → Generate → Save → Embeddings →
Similarity Matrix → Consistency Score → Structural Metrics → Diversity Metrics →
Visualizations → Final Report → Export


## 1. Install Libraries

In [1]:
%pip install -q groq google-generativeai python-dotenv


Note: you may need to restart the kernel to use updated packages.


## 2. Import necessary libraries


In [3]:
import os
import re
import json
import time
import math
import getpass
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")


In [4]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") 

groq_client = None
gemini_model_client = None

if GROQ_API_KEY:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)

if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)

print("Groq configured:", groq_client is not None)
print("Gemini configured:", GEMINI_API_KEY != "")

Groq configured: True
Gemini configured: True


## Defining the prompt and Running configuration

In [6]:
PROVIDER = "groq"                       # "groq" or "gemini"
GROQ_MODEL = "llama-3.1-8b-instant"
GEMINI_MODEL = "gemini-2.5-flash-lite"

N_RUNS = 10
TEMPERATURE = 0.7                       
MAX_TOKENS = 512

PROMPT = """Priya is preparing rasmalai for a family gathering. She makes 3 batches.
Each batch uses 8 pieces of chhena and yields 12 rasmalai. She gives away 15 rasmalai to
neighbors and eats 2 herself while cooking. Her family of 6 then splits the rest equally.

How many rasmalai does each family member get?

Think step by step, then give your final answer as: "Final Answer: <number>"."""

print(f"Provider: {PROVIDER} | Runs: {N_RUNS} | Temperature: {TEMPERATURE}")
print(" PROMPT =>")
print(PROMPT)

Provider: groq | Runs: 10 | Temperature: 0.7
 PROMPT =>
Priya is preparing rasmalai for a family gathering. She makes 3 batches.
Each batch uses 8 pieces of chhena and yields 12 rasmalai. She gives away 15 rasmalai to
neighbors and eats 2 herself while cooking. Her family of 6 then splits the rest equally.

How many rasmalai does each family member get?

Think step by step, then give your final answer as: "Final Answer: <number>".


## Generating responses

In [7]:
def call_groq(prompt, model=GROQ_MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


def call_gemini(prompt, model=GEMINI_MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    m = genai.GenerativeModel(model)
    resp = m.generate_content(
        prompt,
        generation_config={"temperature": temperature, "max_output_tokens": max_tokens},
    )
    return resp.text.strip()


def generate_responses(prompt, n_runs=N_RUNS, provider=PROVIDER, retries=3, sleep_s=1.5):
    """Call the model n_runs times with the identical prompt. Treat API errors as
    data-quality issues: retry, never silently drop a run."""
    call_fn = call_groq if provider == "groq" else call_gemini
    responses = []
    for i in range(n_runs):
        for attempt in range(1, retries + 1):
            try:
                text = call_fn(prompt)
                responses.append(text)
                print(f"Run {i+1}/{n_runs} OK ({len(text)} chars)")
                break
            except Exception as e:
                print(f"Run {i+1}/{n_runs} attempt {attempt} failed: {e}")
                if attempt == retries:
                    responses.append(None)
                    print(f"Run {i+1}/{n_runs} FAILED after {retries} attempts -> stored as None")
                else:
                    time.sleep(sleep_s)
        time.sleep(0.3)  # be polite to rate limits
    return responses


raw_responses = generate_responses(PROMPT, N_RUNS, PROVIDER)
n_failed = sum(r is None for r in raw_responses)
print(f"\nCompleted: {len(raw_responses) - n_failed}/{N_RUNS} succeeded, {n_failed} failed")

Run 1/10 OK (1005 chars)
Run 2/10 OK (1217 chars)
Run 3/10 OK (880 chars)
Run 4/10 OK (1206 chars)
Run 5/10 OK (1305 chars)
Run 6/10 OK (1014 chars)
Run 7/10 OK (1457 chars)
Run 8/10 OK (1597 chars)
Run 9/10 OK (1264 chars)
Run 10/10 OK (1010 chars)

Completed: 10/10 succeeded, 0 failed


### Save Responses

Saved before any analysis, so a bug downstream never costs the API calls 

In [8]:
os.makedirs("outputs", exist_ok=True)

run_metadata = {
    "timestamp": datetime.now().isoformat(),
    "provider": PROVIDER,
    "model": GROQ_MODEL if PROVIDER == "groq" else GEMINI_MODEL,
    "temperature": TEMPERATURE,
    "n_runs": N_RUNS,
    "prompt": PROMPT,
}

raw_record = {"metadata": run_metadata, "responses": raw_responses}
with open("outputs/raw_responses.json", "w") as f:
    json.dump(raw_record, f, indent=2)

# Drop failed (None) runs for analysis but keep the record of how many failed
responses = [r for r in raw_responses if r is not None]

df = pd.DataFrame({"run_id": range(1, len(responses) + 1), "response": responses})
df["length_chars"] = df["response"].str.len()
df["length_words"] = df["response"].str.split().apply(len)

print(f"Saved {len(raw_responses)} raw responses to outputs/raw_responses.json")
df.head()

Saved 10 raw responses to outputs/raw_responses.json


,run_id,response,length_chars,length_words
0,1,"To find out how many rasmalai Priya has left, ...",1005,194
1,2,To determine how many rasmalai each family mem...,1217,237
2,3,To find out how many rasmalai each family memb...,880,170
3,4,To find out how many rasmalai each family memb...,1206,217
4,5,To find out how many rasmalai each family memb...,1305,244


## Commputinng the embeddings

In [9]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(df["response"].tolist(), show_progress_bar=False)

print("Embeddings shape:", embeddings.shape)

c:\Users\quick\anaconda3\envs\computer_vision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 955.65it/s]


Embeddings shape: (10, 384)
